# Data Preparation
- Youtube API setting up and extracting data from a baseline youtube channel.
- Extracting, cleaning, preprocessing and exporting the clean data for model training.
- Using @SundaySarthak as baseline Youtube channel.

## Data Extraction

In [2]:
# importing libraries
import html
import json
import os
import re
from dotenv import load_dotenv
from deep_translator import GoogleTranslator
from googleapiclient.discovery import build
from langdetect import DetectorFactory, detect
import pandas as pd
from tqdm import tqdm

# Ensure reproducible language detection
DetectorFactory.seed = 0

# Load API key from .env 
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found — check your .env file exists and is loaded correctly")

In [3]:
# youtube object
youtube = build("youtube", "v3", developerKey=API_KEY)

# channel detail request
channel_handle = "@SundaySarthak"
response = (
    youtube.channels().list(part="snippet,contentDetails,statistics", forHandle=channel_handle).execute()
)

# Key information
item = response["items"][0]
print(f"Channel Name : {item['snippet']['title']}")
print(f"Channel ID : {item['id']}")
print(f"Subscribers : {item['statistics']['subscriberCount']}")

uploads_playlist_id = item["contentDetails"]["relatedPlaylists"]["uploads"]
print(f"Upload ID : {uploads_playlist_id}")

Channel Name : Sarthak Goswami
Channel ID : UC5fcjujOsqD-126Chn_BAuA
Subscribers : 1900000
Upload ID : UU5fcjujOsqD-126Chn_BAuA


In [4]:
# extracting recent 50 videos ids
playlist_items = []
page_token = None
print("Fetching last 50 uploads from playlist ...")

while len(playlist_items) < 50:
    response = (
        youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=min(50, 50 - len(playlist_items)),
            pageToken=page_token
        ).execute()
    )

    for item in response.get("items", []):
        playlist_items.append({
            "video_id": item["contentDetails"]["videoId"],
            "title": item["snippet"]["title"],
        })

    page_token = response.get("nextPageToken")
    if not page_token:
        break

print(f"Collected videos ids title {len(playlist_items)} videos")

Fetching last 50 uploads from playlist ...
Collected videos ids title 50 videos


In [5]:
# Extracting  video metadata: stats + tags, publish date, category
video_ids = [v["video_id"] for v in playlist_items]

stats_response = (
    youtube.videos().list(part="statistics,snippet", id=",".join(video_ids)).execute()
)

stats_by_id = {item["id"]: item for item in stats_response.get("items", [])}

for video in playlist_items:
    stat_item = stats_by_id.get(video["video_id"], {})
    stats = stat_item.get("statistics", {})
    snippet = stat_item.get("snippet", {})

    video["views"] = int(stats.get("viewCount", 0))
    video["likes"] = int(stats.get("likeCount", 0))
    video["comments"] = int(stats.get("commentCount", 0))
    video["tags"] = snippet.get("tags", [])
    video["published_at"] = snippet.get("publishedAt", None)
    video["category_id"] = snippet.get("categoryId", None)
    video["description"] = snippet.get("description", "")

print(f"Videos metadata and stats for {len(playlist_items)} videos.")

Videos metadata and stats for 50 videos.


In [7]:
raw_comments = []
per_vid_cap = 150

# Sampling both "relevance" (top/most-liked comments) and "time" (most
# recent comments) per video, rather than just one order, to avoid bias
# toward only highly-upvoted comments — recent comments are more likely
# to include newer slang/toxicity patterns not yet reflected in top comments.
for video in tqdm(playlist_items, desc="Extracting Videos"):
    for sort_order in ["relevance", "time"]:
        c_page_token = None
        fetched_in_mode = 0

        while fetched_in_mode < per_vid_cap:
            try:
                c_response = (
                    youtube.commentThreads()
                    .list(
                        part="snippet",
                        videoId=video["video_id"],
                        order=sort_order,
                        maxResults=min(100, per_vid_cap - fetched_in_mode),
                        pageToken=c_page_token
                    ).execute()
                )

                items = c_response.get("items", [])
                if not items:
                    break

                for item in items:
                    top_comment = item["snippet"]["topLevelComment"]["snippet"]
                    raw_comments.append({
                        "video_id": video["video_id"],
                        "comment_id": item["snippet"]["topLevelComment"]["id"],
                        "raw_text": top_comment.get("textOriginal"),
                        "like_count": top_comment.get("likeCount", 0),
                        "reply_count": item["snippet"].get("totalReplyCount", 0),
                        "published_at": top_comment.get("publishedAt"),
                        "sampling_mode": sort_order,
                    })
                    fetched_in_mode += 1
                    if fetched_in_mode >= per_vid_cap:
                        break

                c_page_token = c_response.get("nextPageToken")
                if not c_page_token:
                    break

            except Exception as e:
                # Some videos have comments disabled — this is expected,
                print(f"Skipping {video['video_id']} ({sort_order}): {e}")
                break

Extracting Videos:   0%|          | 0/50 [00:00<?, ?it/s]

Extracting Videos: 100%|██████████| 50/50 [00:44<00:00,  1.12it/s]


In [8]:
# Saving raw comments data 

if raw_comments:
    os.makedirs("../data/raw", exist_ok=True)
    raw_df = pd.DataFrame(raw_comments)
    raw_df.to_csv("../data/raw/raw_comment.csv", index=False, encoding="utf-8")

    print(f"\nExtraction complete! Saved {len(raw_df)} raw comments across {len(playlist_items)} videos.")
    print(f"Sampling distribution:\n{raw_df['sampling_mode'].value_counts()}")
else:
    print("\nNo comments were extracted. Check the error warnings printed above.")


Extraction complete! Saved 14888 raw comments across 50 videos.
Sampling distribution:
sampling_mode
time         7446
relevance    7442
Name: count, dtype: int64


In [9]:
# Saving raw videos metadata 
video_df = pd.DataFrame(playlist_items)
video_df.to_csv("../data/raw/raw_videos.csv", index=False, encoding="utf-8")
print(f"Saved metadata for {len(video_df)} videos.")

Saved metadata for 50 videos.


## Language Normalization & Text Preprocessing

In [10]:
# importing raw data
raw_df = pd.read_csv("../data/raw/raw_comment.csv")
print(f"Loaded raw records: {len(raw_df)}")

Loaded raw records: 14888


In [11]:
# Cleaning comments of emoji, symbols, hashtags, timestamp, excess spaces
import emoji
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # decoding html symbols
    text = html.unescape(text)
    # emoji to text labels
    text = emoji.replace_emoji(text, replace=" ")
    # strip urls, timestamps, 
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", "", text)
    # normalize excess spaces and line breaks
    text = re.sub(r"\s+", " ", text).strip()
    # Strip hashtags (keep the word, drop the #, since hashtag text can still carry meaning)
    text = re.sub(r"#(\w+)", r"\1", text)
    return text

In [12]:
# comments cleaning
raw_df["cleaned_text"] = raw_df["raw_text"].apply(clean_text)

In [13]:
# deduplicate comments — same commenter can post identical text across
df = raw_df.drop_duplicates(subset=["video_id", "cleaned_text"]).reset_index(drop=True)
print(f"Deduplicated: {len(raw_df)} -> {len(df)} comments")

Deduplicated: 14888 -> 12409 comments


## Comment Language Detection and Tagging

In [14]:
def classify_comment_language(text):
    if len(text) < 3:
        return "too_short"
    try:
        lang = detect(text)
    except Exception:
        lang = "unknown"
    return lang

df["detected_language"] = df["cleaned_text"].apply(classify_comment_language)
df["final_text"] = df["cleaned_text"]  # no translation — use cleaned text as-is

In [15]:
# Languages that plausibly reflect genuine non-English/non-Hinglish content
plausible_non_english = {"hi", "mr", "bn", "pa", "ne", "zh-cn", "vi", "ta", "te", "gu"}

def classify_status(lang):
    if lang == "too_short":
        return "too_short"
    if lang == "en":
        return "english"
    if lang in plausible_non_english:
        return "likely_genuine_non_english"
    return "likely_misdetected_hinglish"

df["language_status"] = df["detected_language"].apply(classify_status)
print(df["language_status"].value_counts())

language_status
english                        6308
likely_misdetected_hinglish    5576
likely_genuine_non_english      460
too_short                        65
Name: count, dtype: int64


In [16]:
# Translate the small "genuinely non-English" bucket 
from deep_translator import GoogleTranslator
import time

def translate_non_english(text):
    try:
        result = GoogleTranslator(source="auto", target="en").translate(text)
        time.sleep(0.3)
        return result
    except Exception:
        time.sleep(0.3)
        return text

non_english_mask = df["language_status"] == "likely_genuine_non_english"
print(f"Translating {non_english_mask.sum()} comments...")

df["final_text"] = df["cleaned_text"]  # default: no translation
df.loc[non_english_mask, "final_text"] = df.loc[non_english_mask, "cleaned_text"].apply(translate_non_english)

print("Done.")

Translating 460 comments...
Done.


In [17]:
# Exporting Final data
df["final_text"] = df["cleaned_text"]

os.makedirs("../data/processed", exist_ok=True)
export_path = "../data/processed/comments_cleaned.csv"
df.to_csv(export_path, index=False, encoding="utf-8")

print(f"Saved {len(df)} processed comments to {export_path}")
print(df["language_status"].value_counts())

Saved 12409 processed comments to ../data/processed/comments_cleaned.csv
language_status
english                        6308
likely_misdetected_hinglish    5576
likely_genuine_non_english      460
too_short                        65
Name: count, dtype: int64


### Observation and Conclusion:
- Scraped 10,567+ comments across 50 videos for the baseline YouTube channel.
- The comment section revealed itself to be multi-lingual. Among these, a large number of comments are in romanized Hindi, i.e. Hinglish (Hindi language written in English script).
- To resolve this, different approaches were tested:
  - Translating all of them to English — `langdetect` was not able to properly detect which comments were Hinglish vs. genuinely other languages, frequently misclassifying romanized Hindi as unrelated languages (Indonesian, Somali, Swahili, etc.).
  - Converting to Devanagari before translating — this also led to inconsistency, since transliteration quality was unreliable (e.g. English loanwords getting mangled).
  - Offline/alternative translators (argostranslate, MyMemory, IndicTrans2) — not viable due to installation failures.
  - Hinglish stopword removal before detection — tested as well, but removed words that were actually useful signal for detection, making results worse rather than better.
- Given this, all comments are passed through as cleaned text, flagged with a `language_status` feature with four categories: `english`, `likely_misdetected_hinglish`, `likely_genuine_non_english`, and `too_short`. This is used later to route each comment to the right model.
- `likely_misdetected_hinglish` (45.7% of comments) is scored using a separate classifier trained specifically on Hinglish offensive/not-offensive data, since the English-trained model has no reliable signal for this text.
- `likely_genuine_non_english` (a small minority, ~3.6% of comments) is translated to English before scoring, since this bucket is small enough that translation is practical here — this would not scale the same way for a channel where non-English content is the majority language.